In [31]:
# Auto-reload modules for development (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Market Data

In [32]:
from data.loader import DataLoader

BACKTEST_DATE = '2019-12-01' # Backtesting for 'Covid Crash'
HORIZON = 12

# LOAD FRED Index DATA 
loader = (
    DataLoader().load_from_fred()
    .subset(n_observations=200,last_observation=BACKTEST_DATE)
    .convert_to_real_prices(anchor_date=BACKTEST_DATE,real_data={'P': 520, 'C': 130, 'D': 50_000})
    .compute_log_returns()
)

# Future data for backtesting
actual = loader.get_future_data(last_observation=BACKTEST_DATE)

INFO: Loading data from FRED API...
INFO: [✓] Loaded 524 observations (1982-06 to 2026-01)
INFO: [✓] Subset: 200 observations (2003-05 to 2019-12)
INFO: [✓] Converted to real prices (anchor: 2019-12-01)
INFO: [✓] Log returns computed: 199 observations


## Scenarios Generation

In [33]:
from scenario import RegimeSwitchingGenerator, StressConfig

generator = RegimeSwitchingGenerator(n_regimes=2).fit(loader)

result = generator.generate(n_scenarios=3000, horizon=HORIZON, seed=42)

reduced = generator.reduce(result, n_clusters=1000)

# Inspect regime characteristics
print(generator.regime_summary().to_string())

generator.plot_scenarios(reduced, actual_data=actual, variable_labels={'D': 'Demand (Tn)', 'P': 'Steel Price (€/Tn)', 'C': 'Scrap Cost (€/Tn)'})

INFO: Running auto order selection...
INFO: [RegimeSwitchingVAR] Running pre-fit analysis...
INFO: [✓] Analysis complete. Recommended order: 1
INFO: [RegimeSwitchingVAR] Fitting with order=1...
INFO: [✓] Model fitted successfully!
INFO: [RegimeSwitchingVAR] Generating 3000 scenarios, horizon=12, start=2020-01
INFO: [✓] Generated 3000 scenarios successfully!
INFO: [RegimeSwitchingVAR] Reducing 3000 -> 1000 scenarios (method=kmedoids)
INFO: [OK] Reduced to 1000 scenarios (67% reduction, 0 stress scenarios)


   regime     label  fraction  persistence  expected_duration_months  avg_volatility    mean_D    mean_P    mean_C
0       0    Normal  0.883249     0.936830                 15.830362        0.039031  0.001253  0.000135 -0.001252
1       1  Stress-1  0.116751     0.728367                  3.681434        0.077119 -0.010954 -0.000054  0.006818


## Stochastic Optimization Model

In [34]:
from optimization import StochasticOptimizationModel
from optimization.stochastic import ModelParameters
from params import RiskProfile
    
stoch_model = StochasticOptimizationModel(solver='highs')

stoch_result = stoch_model.run(
    scenarios=reduced.scenarios,
    prob=reduced.probabilities,
    variable_mapping={'C': 'c_spot', 'P': 'p_sell'},
    params=ModelParameters(),
    risk_profile=RiskProfile(risk_aversion=0.3, cvar_alpha=0.05),
)

stoch_result.plot()

# Prepare actual data with correct column names
future_data = actual.rename(columns={'C': 'c_spot', 'P': 'p_sell'})

# Backtest stochastic model
stoch_bt = stoch_model.backtest(
    decisions=stoch_result,
    actual_data=future_data,
    params=ModelParameters(),
)

stoch_bt.plot()

INFO: [✓] Input validation completed!
INFO: [✓] Parameter dictionaries built!
INFO: [✓] Constraints added!
INFO: [✓] Objective function built!
INFO: [✓] Optimization completed successfully!
INFO: [✓] Results extracted!
INFO: [✓] Full results extracted!


INFO: [✓] Backtesting simulation completed!


## 5. Safety Stock Model (Benchmark)

In [35]:
from optimization import SafetyStockModel, SafetyStockParams

ss_model = SafetyStockModel()

# Traditional planning policy
policy = SafetyStockParams(
    service_level=0.95,        # 95% target fill rate
    fixed_pct=0.60,            # 60% via fixed contracts
    framework_pct=0.25,        # 25% via framework options
    production_smoothing=True, # Level-load production
)

ss_result = ss_model.run(
    scenarios=reduced.scenarios,
    prob=reduced.probabilities,
    variable_mapping={'C': 'c_spot', 'P': 'p_sell'},
    params=ModelParameters(),
    policy=policy,
)

ss_result.plot()


ss_bt = ss_model.backtest(
    decisions=ss_result,
    actual_data=future_data,
    params=ModelParameters(),
    policy=policy,
)

ss_bt.plot()

INFO: [SafetyStockPolicy] Applying safety stock policy...
INFO: [âœ“] Safety stock policy applied. Expected Profit: â‚¬106,571,397


INFO: [SafetyStockPolicy] Running backtest...
INFO: [✓] Backtest complete. Total Profit: €179,340,956
